In [9]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import orca

from mazsim import data_loader, variable_loader

In [10]:
project_dir = Path.cwd().parent
orca.add_injectable("project_dir", project_dir)
orca.run(['load_settings', 'load_data', 'build_networks','register_variables'])

Running step 'load_settings'
Registered injectable: data_dir: data
Registered injectable: output_dir: output
Registered injectable: data_model: data_model.py
Registered injectable: geography_id: block_id
Registered injectable: base_year: 2020
Registered injectable: end_year: 2050
Registered injectable: calibrated: True
Registered injectable: hh_ct_type: subregional
Registered injectable: job_ct_type: subregional
Registered injectable: housing_vacancy_rate: 0.05
Registered injectable: output_tables: ['households', 'persons', 'jobs', 'housing_units']
Registered injectable: custom_steps_dir: configs
Registered injectable: custom_steps_files: ['custom_steps.py']
Registered injectable: custom_variables_dir: configs
Registered injectable: custom_variables_files: ['custom_variables.py']
Registered injectable: submodel_groups: {'hlcm': {'ct_type': 'hh_ct_type'}, 'hulcm': {'ct_type': 'hh_ct_type', 'unsegmented_injectable': 'reg_hulcm_step_names'}, 'jlcm': {'ct_type': 'job_ct_type', 'unsegmented

In [11]:
from urbansim.models import util
from urbansim_templates import modelmanager as mm
from urbansim_templates.models import OLSRegressionStep

from urbansim.models.util import (columns_in_filters, columns_in_formula)
from choicemodels.tools import MergedChoiceTable

mm.initialize(Path.joinpath(project_dir, "configs"))

No files from ModelManager 0.1.dev8 or later found in path 'c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs'


In [12]:
# Set explanatory variables
expl_vars = [
    'major_road_node_sum_800_flat',
    'st_ln_zones_total_jobs_15_minutes_am_single_vehicle_to_work_travel_time',
    'st_prop_aggr_sector_id_1_ave_1600_flat', 
    'st_prop_aggr_sector_id_2_ave_1600_flat', 
    'st_prop_aggr_sector_id_3_ave_1600_flat',
    'st_prop_aggr_sector_id_4_ave_1600_flat',
    'st_prop_aggr_sector_id_5_ave_1600_flat',
    'st_ln_block_groups_mean_res_value', 
    'st_ln_block_groups_mean_res_rent',
    'st_total_households_sum_2000_linear',
    'st_block_groups_median_year_built',
    'st_is_transit_stop_sum_1600_flat',
    'st_block_groups_median_income',
    'st_block_groups_prop_built_after_2010_1', 
    'st_prop_children_0_ave_400_flat',
    'st_sum_income_ave_1600_flat'
]

In [ ]:
# housing values
m=OLSRegressionStep()
m.tables=['blocks']
m.filters='(res_value > 0)&(value_impute==0)'

model_spec={'left_side':'res_value',
            'right_side':expl_vars}

m.model_expression=util.str_model_expression(model_spec)
m.out_filters='(all_blocks == 1)'
m.name='hupm_value'
m.out_column='res_value'
m.fit()

Disaggregating prop_built_after_2010_1 to blocks from block_groups
Disaggregating block_group_id to housing_units from blocks
Calculating proportion built_after_2010 1 for block_groups
Calculating number of housing_units for block_groups
Calculating proportion aggr_sector_id 1 for blocks
Calculating number of jobs for blocks
Disaggregating mean_res_rent to blocks from block_groups
Calculating mean_res_rent of blocks for block_groups
Calculating proportion aggr_sector_id 3 for blocks
Calculating proportion aggr_sector_id 4 for blocks
Calculating proportion children 0 for blocks
Calculating number of households for blocks
Disaggregating median_year_built to blocks from block_groups
Calculating median_year_built of housing_units for block_groups
Calculating proportion aggr_sector_id 2 for blocks
Calculating proportion aggr_sector_id 5 for blocks
Disaggregating mean_res_value to blocks from block_groups
Calculating mean_res_value of blocks for block_groups
Calculating sum_income of househo

In [14]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'hupm_value1.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'hupm_value1'


In [ ]:
# housing rent
m=OLSRegressionStep()
m.tables=['blocks']
m.filters='(res_rent > 0)&(rent_impute==0)'

model_spec={'left_side':'res_rent',
            'right_side':expl_vars}

m.model_expression=util.str_model_expression(model_spec)
m.out_filters='(all_blocks == 1)'
m.name='hupm_rent'
m.out_column='res_rent'
m.fit()

                            OLS Regression Results                            
Dep. Variable:               res_rent   R-squared:                       0.768
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     4609.
Date:                Wed, 16 Sep 2026   Prob (F-statistic):               0.00
Time:                        12:55:45   Log-Likelihood:            -1.5854e+05
No. Observations:               22254   AIC:                         3.171e+05
Df Residuals:                   22237   BIC:                         3.173e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                                                                              coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------

In [16]:
# register model (will overwrite previous model)
mm.register(m)

Saving 'hupm_rent1.yaml': c:\Users\jkolberg\PythonProjects\PSRC\psrc_mazsim\projects\baseline2023\configs
Registering model step 'hupm_rent1'
